# 07. 시퀀스 모델: LSTM과 Transformer

문자 단위 언어 모델을 학습해 텍스트를 생성합니다. 이전 문자들을 보고 다음 문자를 예측하는 같은 문제를 LSTM과 Transformer 두 가지로 풉니다.

**학습 목표**
- 텍스트를 정수 시퀀스로 바꾸고 (입력, 다음 문자) 학습 쌍을 만든다
- `nn.Embedding` + `nn.LSTM`으로 순환 모델을 만든다
- `nn.TransformerEncoder`와 causal mask로 자기회귀 모델을 만든다
- 학습된 모델로 텍스트를 샘플링한다

데이터: `data/corpus.txt` (셰익스피어 『햄릿』 독백, 퍼블릭 도메인)

In [ ]:
import torch

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}, device: {device}")

## 1. 데이터 준비

In [ ]:
from pathlib import Path

DATA = Path("data")
corpus_path = DATA / "corpus.txt"

if not corpus_path.exists():
    raise FileNotFoundError(f"{corpus_path}가 없습니다. 저장소에서 data/corpus.txt를 복원하세요.")

text = corpus_path.read_text(encoding="utf-8")
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
vocab_size = len(chars)

def encode(s):
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

data = encode(text)
print(f"문자 수 {len(text)} | 어휘 크기 {vocab_size}")
print(repr(text[:80]))

## 2. 학습 배치 만들기

길이 `SEQ_LEN`인 구간을 무작위로 잘라 입력으로 쓰고, 한 칸 뒤로 민 구간을 정답으로 씁니다.

In [ ]:
SEQ_LEN = 64
BATCH = 64

def get_batch():
    idx = torch.randint(0, len(data) - SEQ_LEN - 1, (BATCH,))
    x = torch.stack([data[i:i + SEQ_LEN] for i in idx])
    y = torch.stack([data[i + 1:i + SEQ_LEN + 1] for i in idx])
    return x.to(device), y.to(device)

xb, yb = get_batch()
print("입력:", repr(decode(xb[0][:30])))
print("정답:", repr(decode(yb[0][:30])))

## 3. 공통 학습/생성 함수

두 모델 모두 `(batch, seq)` 정수를 받아 `(batch, seq, vocab)` 로짓을 반환하도록 만들어 같은 함수를 씁니다.

In [ ]:
from torch import nn
import torch.nn.functional as F

def train(model, steps=1500, lr=3e-3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    model.train()
    for step in range(steps):
        x, y = get_batch()
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
        if step % 300 == 0 or step == steps - 1:
            print(f"step {step:4d} | loss {loss.item():.4f}")
    return losses

@torch.no_grad()
def generate(model, prompt, length=200, temperature=0.8):
    model.eval()
    ids = encode(prompt).unsqueeze(0).to(device)
    for _ in range(length):
        logits = model(ids[:, -SEQ_LEN:])[:, -1] / temperature
        next_id = torch.multinomial(F.softmax(logits, dim=-1), 1)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].cpu())

## 4. LSTM 언어 모델

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, emb=64, hidden=256, layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb)
        self.lstm = nn.LSTM(emb, hidden, num_layers=layers, batch_first=True, dropout=0.1)
        self.head = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        out, _ = self.lstm(self.embed(x))
        return self.head(out)

lstm = CharLSTM(vocab_size).to(device)
print("파라미터 수:", sum(p.numel() for p in lstm.parameters()))
lstm_losses = train(lstm)

In [ ]:
print(generate(lstm, "To be"))

## 5. Transformer 언어 모델

Transformer는 순서 정보가 없으므로 위치 임베딩을 더하고, 미래 문자를 보지 못하도록 **causal mask**(상삼각 마스크)를 사용합니다.

In [ ]:
class CharTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, layers=3, max_len=SEQ_LEN):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=4 * d_model,
                                           dropout=0.1, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        T = x.size(1)
        h = self.embed(x) + self.pos(torch.arange(T, device=x.device))
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        return self.head(self.encoder(h, mask=mask, is_causal=True))

transformer = CharTransformer(vocab_size).to(device)
print("파라미터 수:", sum(p.numel() for p in transformer.parameters()))
tf_losses = train(transformer, lr=1e-3)

In [ ]:
print(generate(transformer, "To be"))

## 6. 학습 곡선 비교

In [ ]:
import matplotlib.pyplot as plt

def smooth(xs, k=50):
    return [sum(xs[max(0, i - k):i + 1]) / len(xs[max(0, i - k):i + 1]) for i in range(len(xs))]

plt.figure(figsize=(6, 3.5))
plt.plot(smooth(lstm_losses), label="LSTM")
plt.plot(smooth(tf_losses), label="Transformer")
plt.xlabel("step")
plt.ylabel("cross entropy (smoothed)")
plt.legend()
plt.show()

## 요약
- 문자 → 정수 → `nn.Embedding`으로 텍스트를 신경망 입력으로 바꾼다
- LSTM은 순서대로 은닉 상태를 전달하고, Transformer는 위치 임베딩 + causal mask + self-attention으로 순서를 다룬다
- 같은 학습/생성 함수로 두 모델을 비교할 수 있다
- 말뭉치가 아주 작으므로 모델이 원문을 거의 외우게 된다. 더 큰 텍스트를 `data/corpus.txt`에 넣으면 일반화된 생성 결과를 볼 수 있다